In [2]:
# Install required packages
!pip install -q transformers datasets seqeval

In [3]:
# Parse CoNLL format
def read_conll(file_path):
    sentences = []
    labels = []
    with open(file_path, encoding='utf-8') as f:
        tokens = []
        tags = []
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                token, tag = line.split()
                tokens.append(token)
                tags.append(tag)
    return sentences, labels

# Replace this path with your uploaded file path
file_path = "/content/amharic_ner_conll_format.txt"
sentences, tags = read_conll(file_path)

In [4]:
# Convert to Hugging Face Dataset format
from datasets import Dataset

dataset = Dataset.from_dict({
    "tokens": sentences,
    "ner_tags": tags
})
label_list = list(set(tag for seq in tags for tag in seq))
label_list.sort()
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [5]:
# Tokenize and align labels
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Davlan/afro-xlmr-base")

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, padding=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_str = label[word_idx]
                if word_idx != prev_word_idx:
                    label_ids.append(label2id[label_str])
                else:
                    if label_str.startswith("B-"):
                        label_str = label_str.replace("B-", "I-")
                    label_ids.append(label2id[label_str])
                prev_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/49 [00:00<?, ? examples/s]

In [6]:
# Fine-tune model
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "Davlan/afro-xlmr-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./ner_model",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_steps=10_000,
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at Davlan/afro-xlmr-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-6-4010436406.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: zumihibet2 (zumihibet2-addis-ababa-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


TrainOutput(global_step=21, training_loss=0.6691663378760928, metrics={'train_runtime': 636.0373, 'train_samples_per_second': 0.231, 'train_steps_per_second': 0.033, 'total_flos': 18981278970192.0, 'train_loss': 0.6691663378760928, 'epoch': 3.0})

In [7]:
trainer.save_model("/content/amharic-ner-afroxlmr")
tokenizer.save_pretrained("/content/amharic-ner-afroxlmr")

('/content/amharic-ner-afroxlmr/tokenizer_config.json',
 '/content/amharic-ner-afroxlmr/special_tokens_map.json',
 '/content/amharic-ner-afroxlmr/sentencepiece.bpe.model',
 '/content/amharic-ner-afroxlmr/added_tokens.json',
 '/content/amharic-ner-afroxlmr/tokenizer.json')